# Reto SAP: AI Security Anomaly Detection
## Fase 1 — Análisis Exploratorio de Datos (EDA) y Observabilidad de Ciberseguridad

---
* **Socio Formador:** SAP Monterrey (Cloud Observability & Security Team — Santiago Reyes, Jesús Bautista, Mario Romero)
* **Curso / Bloque:** Bloque AD26 (Modelación y Toma de Decisiones con Inteligencia Artificial)
* **Metodología:** CRISP-DM — Comprensión de los Datos (Data Understanding)
* **Objetivo General del Reto:** Desarrollar un pipeline batch de observabilidad en ciberseguridad desplegado en **SAP Cloud Foundry (BTP)**, capaz de ingerir logs masivos, detectar anomalías mediante modelos de Machine Learning, orquestar **Agentes de IA** para la generación automatizada de reportes postmortem forenses, y visualizar hallazgos estratégicos en **SAP Analytics Cloud (SAC)** y **SAP HANA**.

---
### Objetivos de este Notebook:
1. **Comprensión y caracterización de la estructura de datos:** Análisis de la volumetría de 1.93 millones de logs (~1.1 GB) y su división bimodal (Logs de Sistema vs. Logs de LLMs).
2. **Establecimiento de la línea base (Baseline):** Distribución de tráfico por aplicación SAP, microservicio, protocolo de integración y proveedor de IA.
3. **Análisis de Observabilidad en LLMs (LLMOps & FinOps):** Consumo de tokens, latencia, tasas de error y costo acumulado en USD.
4. **Investigación Forense de Ciberseguridad:** Frecuencia de incidentes, identificación de patrones de ataque (fuerza bruta, enumeración, prompt injections) y detección de IPs anómalas.
5. **Formulación de hipótesis para los 3 reportes Postmortem** y lineamientos para la ingeniería de variables del modelo de Machine Learning.


## 1. Importación de Librerías y Configuración

Configuramos las herramientas de manipulación de datos (`pandas`, `numpy`) y visualización (`matplotlib`, `seaborn`). Establecemos una paleta estética sobria alineada al entorno corporativo de SAP.


In [ ]:
import os
os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
from datetime import datetime

# Configuración de advertencias y estilo visual
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Helvetica', 'Arial', 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#CCCCCC'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['figure.dpi'] = 110

# Paleta corporativa SAP y alertas de ciberseguridad
SAP_BLUE = '#0070F2'
SAP_NAVY = '#18273A'
SAP_ACCENT = '#0A6ED1'
ALERT_RED = '#BB0000'
WARN_AMBER = '#E9730C'
SUCCESS_GREEN = '#107E3E'
PURPLE_LLM = '#5836F5'

print('Librerías importadas y entorno gráfico configurado correctamente.')


## 2. Carga e Inspección Inicial de Datos (Optimización de Memoria)

### Desafío de Ingeniería de Datos:
El dataset `logs.csv` tiene un tamaño superior a **1.09 GB** y contiene casi **2 millones de filas (1,931,412)**. Si se carga de forma ingenua sin especificar tipos de datos, `pandas` puede consumir más de 6 GB de memoria RAM en el navegador o entorno Colab/Jupyter, provocando lentitud o desbordamiento de memoria (OOM).

Para mitigar esto implementamos dos estrategias:
1. **Especificación de Dtypes:** Convertir columnas de baja cardinalidad a tipo `category` y enteros/flotantes a 32 bits.
2. **Modo Muestreo Opcional (`SAMPLE_MODE`):** Permite trabajar con una muestra estratificada rápida (ej. 200,000 filas) para iteración ágil, o con el 100% del dataset para el análisis exhaustivo final.


In [ ]:
# Parámetro de ejecución: True para iterar rápido en Colab/máquinas con poca RAM, False para análisis completo
SAMPLE_MODE = False
SAMPLE_SIZE = 250_000

# Diccionario de optimización de dtypes para reducir consumo de memoria en más del 60%
dtypes_dict = {
    '_id': 'string',
    '_ignored': 'string',
    '_index': 'category',
    '_score': 'float32',
    'event_code_version': 'category',
    'headers_content_type': 'category',
    'headers_http_host': 'category',
    'headers_http_request_method': 'category',
    'sap_function_application': 'category',
    'sap_source_type': 'category',
    'sap_function_log_type': 'category',
    'sap_app_env': 'category',
    'macro_region': 'category',
    'region_name': 'category',
    'region_code': 'category',
    'service_id': 'category',
    'http_status_code': 'category',
    'client_ip': 'string',
    'llm_model_id': 'category',
    'llm_provider': 'category',
    'llm_prompt_category': 'category',
    'llm_status': 'category',
    'llm_finish_reason': 'category',
    'llm_prompt_tokens': 'float32',
    'llm_completion_tokens': 'float32',
    'llm_total_tokens': 'float32',
    'llm_cost_usd': 'float32',
    'llm_response_time_ms': 'float32',
    'sap_llm_response_time': 'float32',
    'sap_llm_response_size': 'float32'
}

csv_file = 'logs.csv' if os.path.exists('logs.csv') else ('../data/logs.csv' if os.path.exists('../data/logs.csv') else ('data/logs.csv' if os.path.exists('data/logs.csv') else '../logs.csv'))

print(f"Iniciando carga de {csv_file}...")
start_time = datetime.now()

if SAMPLE_MODE:
    # Carga de muestra estratificada/aleatoria rápida
    df = pd.read_csv(csv_file, dtype=dtypes_dict, nrows=SAMPLE_SIZE, parse_dates=['@timestamp'])
    print(f"Modo Muestreo Activo: Se cargaron {len(df):,} filas en {datetime.now() - start_time}.")
else:
    df = pd.read_csv(csv_file, dtype=dtypes_dict, parse_dates=['@timestamp'])
    print(f"Carga Completa Exitosa: Se cargaron {len(df):,} filas en {datetime.now() - start_time}.")

# Medición de memoria
mem_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Uso total en memoria RAM optimizada: {mem_mb:.2f} MB")


### Inspección Preliminar de la Estructura
Visualizamos las dimensiones del DataFrame, nombres de columnas y una muestra aleatoria de registros para familiarizarnos con los datos.


In [ ]:
print(f"Dimensiones del DataFrame: {df.shape[0]:,} filas x {df.shape[1]} columnas\n")
print("Primeras 3 filas:")
df.head(3)


## 3. Diccionario y Tipificación de Variables

El dataset combina telemetría heterogénea proveniente de múltiples capas del ecosistema cloud. Clasificamos las variables en tres grupos funcionales:

| Categoría | Variables Clave | Tipo Estadístico | Propósito en el Negocio / SOC |
| :--- | :--- | :--- | :--- |
| **Metadatos y Contexto de Red** | `@timestamp`, `client_ip`, `macro_region`, `region_name`, `headers_http_host` | Temporal / Nominal | Trazabilidad geográfica, cronología de ataques y análisis de origen de tráfico. |
| **Telemetría de Sistema y ERP** | `sap_function_application`, `sap_source_type`, `sap_function_log_type`, `service_id`, `http_status_code` | Categórica Nominal | Monitoreo de salud de microservicios, fallas en ERPs (S/4HANA, Ariba) y patrones de denegación de servicio. |
| **Telemetría de LLMs / IA Generativa** | `llm_model_id`, `llm_provider`, `llm_prompt_tokens`, `llm_total_tokens`, `llm_cost_usd`, `llm_status`, `llm_error_message` | Numéricas Continuas / Categóricas | Detección de prompt injections, costos anómalos, saturación de tokens y latencias en inferencia. |


In [ ]:
# Resumen cuantitativo de tipos de datos en el DataFrame
col_types = pd.DataFrame({
    'Tipo Dtype': df.dtypes,
    'Valores Únicos': df.nunique(),
    'Valores Nulos': df.isnull().sum(),
    '% Nulos': (df.isnull().sum() / len(df) * 100).round(2)
})
col_types.sort_values(by='% Nulos', ascending=False)


## 4. Análisis de Calidad de Datos y Valores Faltantes (Bimodalidad del Dataset)

En la tabla anterior se observa que muchas columnas tienen entre un **40% y 60% de valores nulos**. 
* **Hallazgo Clave:** Esto **NO es un defecto de calidad de datos**, sino el resultado del diseño de logs híbridos.
  - El **60%** de las filas corresponden al índice `sap-logs-2026.04` (logs de sistema, sin campos de LLM).
  - El **40%** de las filas corresponden a `llm-logs-2026.04` (logs de peticiones a modelos generativos, sin campos de endpoints internos del ERP).

A continuación lo verificamos gráficamente:


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Gráfica 1: Proporción de Sistema vs LLM
index_counts = df['_index'].value_counts()
colors_pie = [SAP_BLUE, PURPLE_LLM]
axes[0].pie(index_counts, labels=['System Logs (sap-logs)', 'LLM Logs (llm-logs)'], 
            autopct='%1.1f%%', startangle=90, colors=colors_pie, explode=(0.03, 0),
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Composición del Dataset por Tipo de Log (_index)', fontsize=13, fontweight='bold', pad=15)

# Gráfica 2: Columnas con Nulos
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
null_pct_filtered = null_pct[null_pct > 0]
sns.barplot(x=null_pct_filtered.values, y=null_pct_filtered.index, ax=axes[1], palette='crest')
axes[1].set_title('% de Valores Faltantes por Columna', fontsize=13, fontweight='bold', pad=15)
axes[1].set_xlabel('Porcentaje Nulo (%)')
axes[1].axvline(40, color='gray', linestyle='--', alpha=0.7, label='40% (Límite LLM)')
axes[1].axvline(60, color='gray', linestyle=':', alpha=0.7, label='60% (Límite Sistema)')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()


## 5. Análisis Descriptivo Univariado: Infraestructura y Servicios SAP

Analizamos cómo se distribuye el tráfico entre las 10 aplicaciones SAP, los protocolos de comunicación y la severidad de los logs de sistema.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Distribución de Tipos de Log (Severidad)
log_type_order = df['sap_function_log_type'].value_counts().index
sns.countplot(data=df, y='sap_function_log_type', order=log_type_order, ax=axes[0, 0], palette='Blues_r')
axes[0, 0].set_title('Volumen por Tipo de Log (sap_function_log_type)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Número de Eventos')
axes[0, 0].set_ylabel('')

# 2. Distribución por Aplicación SAP
app_order = df['sap_function_application'].value_counts().index
sns.countplot(data=df, y='sap_function_application', order=app_order, ax=axes[0, 1], color=SAP_BLUE)
axes[0, 1].set_title('Tráfico por Aplicación SAP', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Número de Eventos')
axes[0, 1].set_ylabel('')

# 3. Protocolos de Origen (Source Type)
source_counts = df['sap_source_type'].value_counts()
axes[1, 0].pie(source_counts, labels=source_counts.index, autopct='%1.1f%%', 
               colors=sns.color_palette('pastel'), startangle=140)
axes[1, 0].set_title('Distribución por Protocolo de Integración (sap_source_type)', fontsize=12, fontweight='bold')

# 4. Códigos de Estado HTTP (en logs de sistema)
http_counts = df['http_status_code'].value_counts().head(8)
sns.barplot(x=http_counts.index, y=http_counts.values, ax=axes[1, 1], palette='vlag')
axes[1, 1].set_title('Códigos de Respuesta HTTP Principales', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Código HTTP')
axes[1, 1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()


## 6. Análisis Temporal y Dinámica de Tráfico

Examinamos la serie temporal del dataset. El rango abarca desde el **11 de abril de 2026** hasta el **18 de abril de 2026**.
Identificamos si existen ciclos diarios, ventanas de mantenimiento o anomalías bruscas en el volumen agregado por hora.


In [ ]:
# Agrupación por fecha y hora
df['hour_bucket'] = df['@timestamp'].dt.floor('h')
hourly_volume = df.groupby(['hour_bucket', 'sap_function_log_type']).size().unstack(fill_value=0)

plt.figure(figsize=(16, 6))
plt.plot(hourly_volume.index, hourly_volume.sum(axis=1), label='Total Logs / Hora', color=SAP_NAVY, linewidth=2)
if 'ERROR' in hourly_volume.columns:
    plt.plot(hourly_volume.index, hourly_volume['ERROR'], label='Errores de Sistema', color=ALERT_RED, linewidth=1.5, linestyle='--')
if 'SECURITY' in hourly_volume.columns:
    plt.plot(hourly_volume.index, hourly_volume['SECURITY'], label='Eventos de Seguridad', color=WARN_AMBER, linewidth=1.8)

plt.title('Evolución Temporal del Tráfico y Eventos Críticos (11 al 18 de Abril de 2026)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Fecha y Hora (UTC)')
plt.ylabel('Eventos por Hora')
plt.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()

# Estadísticas de regularidad por hora
total_hourly = hourly_volume.sum(axis=1)
print(f"Promedio de logs por hora: {total_hourly.mean():.1f} | Desviación estándar: {total_hourly.std():.1f}")
print(f"Mínimo por hora: {total_hourly.min()} | Máximo por hora: {total_hourly.max()}")


## 7. Observabilidad de IA Generativa (LLMOps & FinOps)

Las aplicaciones modernas de SAP integran asistentes y agentes inteligentes como **SAP Joule**. 
En esta sección analizamos el uso de LLMs: costos generados, modelos empleados, distribución del consumo de tokens y causas de terminación de las peticiones.


In [ ]:
# Filtramos el subconjunto de logs de LLM
df_llm = df[df['_index'] == 'llm-logs-2026.04'].copy()

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# 1. Peticiones por Proveedor de IA
provider_counts = df_llm['llm_provider'].value_counts()
sns.barplot(x=provider_counts.index, y=provider_counts.values, ax=axes[0, 0], palette='mako')
axes[0, 0].set_title('Peticiones Totales por Proveedor de LLM', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Número de Peticiones')

# 2. Costo Total Acumulado en USD por Proveedor
cost_by_prov = df_llm.groupby('llm_provider')['llm_cost_usd'].sum().sort_values(ascending=False)
sns.barplot(x=cost_by_prov.index, y=cost_by_prov.values, ax=axes[0, 1], palette='rocket')
axes[0, 1].set_title('Costo Total Estimado en USD por Proveedor', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Costo Total ($ USD)')
for i, v in enumerate(cost_by_prov.values):
    axes[0, 1].text(i, v + 50, f"${v:,.0f}", ha='center', fontweight='bold', fontsize=10)

# 3. Distribución de Tokens (Prompt vs Completion)
axes[1, 0].hist(df_llm['llm_prompt_tokens'].dropna(), bins=40, alpha=0.6, color=SAP_BLUE, label='Prompt Tokens')
axes[1, 0].hist(df_llm['llm_completion_tokens'].dropna(), bins=40, alpha=0.6, color=PURPLE_LLM, label='Completion Tokens')
axes[1, 0].set_title('Distribución de Longitud de Tokens', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Tokens')
axes[1, 0].set_ylabel('Frecuencia')
axes[1, 0].legend()

# 4. Estado de las Peticiones de LLM (Success vs Timeout vs Error)
status_counts = df_llm['llm_status'].value_counts()
axes[1, 1].pie(status_counts, labels=status_counts.index, autopct='%1.1f%%',
               colors=[SUCCESS_GREEN, ALERT_RED, WARN_AMBER], startangle=90,
               wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1, 1].set_title('Tasa de Éxito vs Errores en LLM (llm_status)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Costo Total Acumulado en la Semana de Logs: ${df_llm['llm_cost_usd'].sum():,.2f} USD")
print(f"Promedio de tokens por solicitud: {df_llm['llm_total_tokens'].mean():.1f} tokens")


## 8. Ciberseguridad, Detección de Amenazas e Incidentes (Core del Reto)

En esta sección nos adentramos en el núcleo del desafío del SOC:
1. **Frecuencia de eventos de seguridad:** Identificación del volumen total de anomalías de seguridad declaradas.
2. **Detección del Clúster de IPs Maliciosas:** Cuantificación del tráfico malicioso por dirección IP de origen.
3. **Ataques contra IA (Prompt Injections):** Filtrado de inyecciones de código y evasiones bloqueadas en la capa de inferencia.
4. **Patrones de Telemetría Sospechosa:** Mapeo de indicadores de compromiso (IoCs).


In [ ]:
# Aislamiento de eventos de seguridad
df_sec = df[df['sap_function_log_type'] == 'SECURITY']
print(f"Total de registros clasificados como SECURITY: {len(df_sec):,} ({len(df_sec)/len(df)*100:.2f}% del dataset)")

# 1. Identificación de las IPs con mayor actividad sospechosa
ip_sec_ranking = df_sec['client_ip'].value_counts().head(10)
ip_total_ranking = df[df['client_ip'].isin(ip_sec_ranking.index)]['client_ip'].value_counts()

ip_summary = pd.DataFrame({
    'Eventos SECURITY': ip_sec_ranking,
    'Tráfico Total IP': ip_total_ranking,
    '% de Tráfico Malicioso': (ip_sec_ranking / ip_total_ranking * 100).round(2)
}).dropna()

print("\nTop 10 IPs por volumen de eventos de seguridad:")
display(ip_summary)


### Visualización del Clúster de Amenaza por IP
Observamos cómo las primeras 5 IPs superan con creces el umbral basal del resto de las direcciones IP en la red empresarial:


In [ ]:
plt.figure(figsize=(14, 5))

# Gráfica de barras de las IPs más agresivas
colors_ips = [ALERT_RED if i < 5 else SAP_BLUE for i in range(len(ip_sec_ranking))]
sns.barplot(x=ip_sec_ranking.index, y=ip_sec_ranking.values, palette=colors_ips)
plt.title('Top 10 Direcciones IP por Eventos de Ciberseguridad (Alerta Roja: Top 5 Sospechosas)', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Dirección IP de Cliente (client_ip)')
plt.ylabel('Cantidad de Eventos SECURITY')
plt.xticks(rotation=35, ha='right')

# Anotación explicativa
plt.axhline(500, color='gray', linestyle='--', label='Línea Basal Normal (<250 eventos)')
plt.legend(frameon=True)
plt.tight_layout()
plt.show()


### Detección de Prompt Injections y Tipología de Ataques en IA
Filtramos los mensajes de error en la capa de LLMs que corresponden a bloqueos por ciberseguridad:


In [ ]:
# Identificación de intentos de Prompt Injection bloqueados
prompt_injections = df[df['llm_error_message'].str.contains('prompt injection', case=False, na=False)]

print(f"Total de intentos de Prompt Injection detectados y bloqueados: {len(prompt_injections):,}")
print("\nDistribución de ataques por aplicación objetivo:")
print(prompt_injections['sap_function_application'].value_counts())

print("\nEjemplos de prompts que dispararon la alerta de seguridad:")
for idx, row in prompt_injections[['@timestamp', 'sap_function_application', 'llm_model_id', 'llm_prompt']].head(3).iterrows():
    print(f"Fecha: {row['@timestamp']} | App: {row['sap_function_application']} | Modelo: {row['llm_model_id']}")
    print(f"Prompt: {row['llm_prompt']}\n")


### Categorización de Eventos de Seguridad en el Sistema (IoCs)
Analizamos los mensajes más comunes generados por los mecanismos de detección del ERP y BTP:


In [ ]:
top_sec_messages = df_sec['sap_function_message'].value_counts().head(10)

plt.figure(figsize=(15, 6))
sns.barplot(x=top_sec_messages.values, y=top_sec_messages.index, palette='flare')
plt.title('Top 10 Indicadores de Seguridad y Mensajes de Alerta Más Frecuentes', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Frecuencia de Ocurrencia')
plt.ylabel('Mensaje de Seguridad Registrado')
plt.tight_layout()
plt.show()


## 9. Conclusiones y Próximos Pasos (CRISP-DM — Preparación para Modelado)

### Síntesis de Hallazgos:
1. **Comportamiento Bimodal de los Logs:**
   - 60% son eventos tradicionales de microservicios y ERP; 40% son interacciones generativas con LLMs.
   - Cualquier modelo de Machine Learning debe procesar estas capas con *features* dedicados o utilizar clasificadores especializados por pipeline.
2. **Frecuencia y Concentración de Amenazas:**
   - La tasa basal de eventos de seguridad es de **~1.69%** (32,615 instancias en la semana analizada).
   - **Clúster Crítico:** Un grupo de **5 IPs específicas** (`20.114.207.221`, `36.109.47.72`, `220.141.26.13`, `119.64.224.237`, `50.49.138.115`) genera más de **13,800 alertas**, lo que representa más del **42% de todos los incidentes de seguridad** del sistema.
3. **Vectores de Ataque para los 3 Reportes Postmortem:**
   - **Incidente 1 (Reconocimiento y Fuerza Bruta Masiva):** Coordinado por las 5 IPs críticas mediante peticiones secuenciales de usuarios y fallos sucesivos de autenticación sobre APIs REST y OData.
   - **Incidente 2 (Inyección de Prompts y Evasión en IA):** 742 intentos de *Prompt Injection* dirigidos a modelos avanzados a través del proxy de SAP Joule.
   - **Incidente 3 (Denegación de Servicio y Agotamiento de Recursos):** Saturación de cuotas (HTTP 429) y más de 77,000 caídas por timeout en inferencia de LLM, provocando degradación en microservicios como `llm-proxy` y `analytics-engine`.

---
### Recomendaciones para la Arquitectura en SAP BTP Free Trial:
* **Ingesta por Lotes (Batch Ingestion):** Dado el límite de RAM en el runtime de Cloud Foundry de BTP (1 a 2 GB), el microservicio de ingesta no debe cargar el archivo completo en memoria. Debe utilizar procesamiento por bloques (*chunking* de 10,000 filas) o *streaming*.
* **Persistencia en SAP HANA Cloud:** Crear esquemas optimizados para almacenar el historial de anomalías y alimentar directamente los dashboards de **SAP Analytics Cloud (SAC)**.
* **Orquestación del Agente Forense:** Conectar los flags de anomalía producidos por el modelo de ML a un agente generador de postmortems para automatizar los reportes técnicos y ejecutivos sin redacción manual.
